In [45]:
from datasets import load_dataset

corpus = load_dataset(
    "BeIR/scidocs",
    "corpus"
)

queries = load_dataset(
    "BeIR/scidocs",
    "queries"
)

import pandas as pd
import numpy as np

corpus_df = corpus["corpus"].to_pandas()
queries_df = queries["queries"].to_pandas()

corpus_df["text_words"] = corpus_df["text"].str.split().str.len()
corpus_df["title_words"] = corpus_df["title"].str.split().str.len()

queries_df["text_words"] = queries_df["text"].str.split().str.len()
queries_df["title_words"] = queries_df["title"].str.split().str.len()

README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

corpus/corpus-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.7MB            

corpus/corpus-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating corpus split:   0%|          | 0/25657 [00:00<?, ? examples/s]

queries/queries-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 92.6kB            

queries/queries-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating queries split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
print(queries_df.head())

                                        _id title  \
0  78495383450e02c5fe817e408726134b3084905d         
1  7dcb308b9292a8bc87d6f7793d2ca5e0e19dfa40         
2  8c872ecd87945e71fcd9fa1b6cb1133cfe805bf2         
3  3a63667284dc8b9687ed1620406030bfe39af3c9         
4  071f47b7bc5830643e31dbed82e0375bf9b26559         

                                                text  text_words  title_words  
0  A Direct Search Method to solve Economic Dispa...          12            0  
1  Bearish-Bullish Sentiment Analysis on Financia...           6            0  
2  Predicting defects in SAP Java code: An experi...           9            0  
3  Active-Metric Learning for Classification of R...           9            0  
4  Ad Hoc Retrieval Experiments Using WordNet and...          10            0  


In [46]:
!pip install -q beir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.9/304.9 kB 20.7 MB/s eta 0:00:00


In [47]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader

dataset = "scidocs"

url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

data_path = util.download_and_unzip(
    url,
    "datasets"
)

corpus_beir, queries_beir, qrels = GenericDataLoader(
    data_folder=data_path
).load(split="test")

datasets/scidocs.zip:   0%|          | 0.00/136M [00:00<?, ?iB/s]

  0%|          | 0/25657 [00:00<?, ?it/s]

In [48]:
import pandas as pd

qrels_records = []
for query_id, doc_scores in qrels.items():
    for doc_id, score in doc_scores.items():
        qrels_records.append({'query_id': query_id, 'doc_id': doc_id, 'score': score})

qrels_df = pd.DataFrame(qrels_records)

qrels_dict = (
    qrels_df.groupby("query_id")["doc_id"]
    .apply(set)
    .to_dict()
)


corpus_id_to_doc_id = dict(
    enumerate(corpus_df["_id"])
)



In [ ]:
print(qrels_df.head())

                                   query_id  \
0  78495383450e02c5fe817e408726134b3084905d   
1  78495383450e02c5fe817e408726134b3084905d   
2  78495383450e02c5fe817e408726134b3084905d   
3  78495383450e02c5fe817e408726134b3084905d   
4  78495383450e02c5fe817e408726134b3084905d   

                                     doc_id  score  
0  632589828c8b9fca2c3a59e97451fde8fa7d188d      1  
1  86e87db2dab958f1bd5877dc7d5b8105d6e31e46      1  
2  2a047d8c4c2a4825e0f0305294e7da14f8de6fd3      1  
3  506172b0e0dd4269bdcfe96dda9ea9d8602bbfb6      1  
4  51317b6082322a96b4570818b7a5ec8b2e330f2f      1  


In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Max sequence length:", model.max_seq_length)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Max sequence length: 256
Embedding dimension: 384


/tmp/ipykernel_1390/3460295810.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [ ]:
corpus_embeddings = model.encode(
    corpus_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print(corpus_embeddings.shape)

Batches:   0%|          | 0/802 [00:00<?, ?it/s]

torch.Size([25657, 384])


In [ ]:
query_embeddings = model.encode(
    queries_df["text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import pandas as pd

save_dir =  "/content/drive/MyDrive/Start_LLM/Enterprise_Semantic_Search_Engine/Data"

train_pairs = pd.read_parquet(f"{save_dir}/train_pairs.parquet")
val_pairs = pd.read_parquet(f"{save_dir}/val_pairs.parquet")
test_qrels = pd.read_parquet(f"{save_dir}/test_qrels.parquet")
hard_negatives = pd.read_parquet(f"{save_dir}/hard_negatives.parquet")

print("train_pairs:", train_pairs.shape)
print("val_pairs:", val_pairs.shape)
print("test_qrels:", test_qrels.shape)
print("hard_negatives:", hard_negatives.shape)

In [ ]:
!pip install faiss-gpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 9.3 MB/s eta 0:00:00


In [ ]:
import faiss
import torch

def build_hnsw_index(corpus_embeddings, M=32, ef_search=256):
    """
    Build an HNSW index for approximate nearest-neighbor search.
    """

    dimension = corpus_embeddings.shape[1]

    index = faiss.IndexHNSWFlat(
        dimension,
        M,
        faiss.METRIC_INNER_PRODUCT,
    )

    index.hnsw.efSearch = ef_search

    # Move embeddings to CPU and convert to numpy if they are on GPU
    if isinstance(corpus_embeddings, torch.Tensor):
        corpus_embeddings = corpus_embeddings.cpu().numpy()

    index.add(corpus_embeddings)

    return index


def search(index, query_embeddings, top_k=10):
    """
    Retrieve top-k nearest documents for each query.
    """

    # Move query embeddings to CPU and convert to numpy if they are on GPU
    if isinstance(query_embeddings, torch.Tensor):
        query_embeddings = query_embeddings.cpu().numpy()

    scores, indices = index.search(
        query_embeddings,
        top_k,
    )

    return scores, indices

In [ ]:
import torch
index = build_hnsw_index(corpus_embeddings)

In [ ]:
scores, indices = search(index, query_embeddings, top_k=10)

In [ ]:
indices[0]

array([    1, 23875, 14045, 20570, 14060, 19958, 10372, 22109, 11732,
       12830])

In [ ]:
scores[0]

array([0.5885634 , 0.4246573 , 0.4246131 , 0.42168707, 0.42166615,
       0.41366702, 0.404986  , 0.40021944, 0.40020597, 0.3988042 ],
      dtype=float32)

In [ ]:
import os
import shutil

# اگر /content/drive وجود دارد و mount نشده، پاکش می‌کنیم
if os.path.exists("/content/drive"):
    shutil.rmtree("/content/drive")

os.makedirs("/content/drive")

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

save_dir =  "/content/drive/MyDrive/Start_LLM/Enterprise_Semantic_Search_Engine/Data"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
import numpy as np

np.save(
    f"{save_dir}/vector_search_top10_doc_ids.npy",
    indices
)

np.save(
    f"{save_dir}/vector_search_top10_scores.npy",
    scores
)

In [ ]:
save_dir =  "/content/drive/MyDrive/Start_LLM/Enterprise_Semantic_Search_Engine/Data"
import numpy as np


indices = np.load(
    f"{save_dir}/vector_search_top10_doc_ids.npy"
)

scores = np.load(
    f"{save_dir}/vector_search_top10_scores.npy"
)

In [ ]:
retrieved_docs = corpus_df.iloc[indices.flatten()]

print("Retrieved docs shape:", retrieved_docs.shape)

Retrieved docs shape: (10000, 5)


In [ ]:
context_candidates = []

for query_idx in range(len(indices)):
    # Get the document indices for the current query
    current_query_doc_indices = indices[query_idx]

    # Retrieve these documents from the original corpus_df as a DataFrame
    query_docs_df = corpus_df.iloc[current_query_doc_indices]

    docs = []

    for rank, (_, doc) in enumerate(query_docs_df.iterrows(), start=1):

        docs.append({
            "rank": rank,
            "doc_id": doc["_id"],
            "title": doc["title"],
            "text": doc["text"],
            "score": scores[query_idx, rank - 1]
        })

    context_candidates.append(docs)

print("Number of queries:", len(context_candidates))
print("Documents per query:", len(context_candidates[0]))

Number of queries: 1000
Documents per query: 10


In [ ]:
np.save(
    f"{save_dir}/context_candidate.npy",
    context_candidates
)

In [ ]:
temp = []

for i in range(len(context_candidates)):
  query_word_counts = [] # Initialize an inner list for each query
  for j in range(len(context_candidates[i])):
    # Access the 'text' field, split it into words, and get the length
    word_count = len(context_candidates[i][j]['text'].split())
    query_word_counts.append(word_count)
  temp.append(query_word_counts)

print("Shape of temp (queries x docs_per_query):", len(temp), "x", len(temp[0]))

Shape of temp (queries x docs_per_query): 1000 x 10


In [ ]:
import pandas as pd

# Flatten the list of lists into a single list of word counts
flat_word_counts = [count for sublist in temp for count in sublist]

# Convert the flat list to a Pandas Series to use .describe()
word_counts_series = pd.Series(flat_word_counts)

print(word_counts_series.describe(percentiles=[0.90, 0.95, 0.99, 0.999]))

count    10000.00000
mean       160.44340
std         98.98264
min          0.00000
50%        148.00000
90%        234.00000
95%        270.00000
99%        407.02000
99.9%     1509.03200
max       1802.00000
dtype: float64


In [ ]:
import numpy as np
import pandas as pd

# The 'temp' variable already contains the word counts for the retrieved documents
# in the desired shape (number of queries x documents per query).
# Convert the list of lists to a NumPy array and sum along axis 1.
top10_context_lengths = np.array(temp).sum(axis=1)

pd.Series(top10_context_lengths).describe(
    percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]
)

,0
count,1000.000000
mean,1604.434000
std,357.148138
min,0.000000
50%,1549.500000
90%,1958.000000
95%,2220.150000
99%,2976.090000
99.9%,3701.272000
max,3973.000000


In [ ]:
import numpy as np
import pandas as pd



relevance_matrix = np.zeros(
    indices.shape,
    dtype=bool
)

for i, query_id in enumerate(qrels_df["query_id"].unique()[:len(indices)]):
    relevant_docs = qrels_dict.get(query_id, set())

    for rank, doc_id in enumerate(indices[i]):
        relevance_matrix[i, rank] = doc_id in relevant_docs

In [ ]:
relevance_matrix.shape

(1000, 10)

In [ ]:
relevant_count_per_query = relevance_matrix.sum(axis=1)

print("Average relevant docs in Top-10:",
      relevant_count_per_query.mean())

print("Median:",
      np.median(relevant_count_per_query))

print("Min:",
      relevant_count_per_query.min())

print("Max:",
      relevant_count_per_query.max())

Average relevant docs in Top-10: 0.0
Median: 0.0
Min: 0
Max: 0


In [ ]:
print(type(qrels))

first_key = next(iter(qrels))
print("First qrels key:", first_key)
print("Relevant docs:", list(qrels[first_key])[:10])

print("\nNumber of queries in qrels:", len(qrels))

print("\nTop-10 shape:", indices.shape)

<class 'dict'>
First qrels key: 78495383450e02c5fe817e408726134b3084905d
Relevant docs: ['632589828c8b9fca2c3a59e97451fde8fa7d188d', '86e87db2dab958f1bd5877dc7d5b8105d6e31e46', '2a047d8c4c2a4825e0f0305294e7da14f8de6fd3', '506172b0e0dd4269bdcfe96dda9ea9d8602bbfb6', '51317b6082322a96b4570818b7a5ec8b2e330f2f', '857a8c6c46b0a85ed6019f5830294872f2f1dcf5', '12f107016fd3d062dff88a00d6b0f5f81f00522d', '1ae0ac5e13134df7a0d670fc08c2b404f1e3803c', '7d3c9c4064b588d5d8c7c0cb398118aac239c71b', '305c45fb798afdad9e6d34505b4195fa37c2ee4f']

Number of queries in qrels: 1000

Top-10 shape: (1000, 10)


In [ ]:
# Convert row indices returned by vector search
# into the actual document IDs used by qrels.

top10_doc_ids_actual = corpus_df.iloc[indices.flatten()]["_id"].values

print(top10_doc_ids_actual.shape)
print(top10_doc_ids_actual[0])

(10000,)
86e87db2dab958f1bd5877dc7d5b8105d6e31e46


In [ ]:
num_queries = len(qrels) # Use len(qrels) as qrels is a dict and its keys are query_ids
top_k = indices.shape[1]

relevance_matrix = np.zeros(
    (num_queries, top_k),
    dtype=bool
)

query_ids = list(qrels.keys()) # Define query_ids here

for i, query_id in enumerate(query_ids):
    relevant_docs = qrels[query_id] # relevant_docs is a dict from doc_id to score

    # Extract the top-k document IDs for the current query from the flattened array
    current_query_top_k_doc_ids = top10_doc_ids_actual[i * top_k : (i + 1) * top_k]

    for rank, doc_id in enumerate(current_query_top_k_doc_ids):
        # Check if doc_id is in the keys of relevant_docs dictionary
        relevance_matrix[i, rank] = doc_id in relevant_docs

In [ ]:
relevant_count_per_query = relevance_matrix.sum(axis=1)

print("Average relevant docs in Top-10:",
      relevant_count_per_query.mean())

print("Median:",
      np.median(relevant_count_per_query))

print("Min:",
      relevant_count_per_query.min())

print("Max:",
      relevant_count_per_query.max())

Average relevant docs in Top-10: 1.086
Median: 1.0
Min: 0
Max: 5


In [ ]:
relevant_by_rank = relevance_matrix.mean(axis=0)

for rank, rate in enumerate(relevant_by_rank, start=1):
    print(f"Rank {rank}: {rate:.4f}")

Rank 1: 0.2210
Rank 2: 0.1740
Rank 3: 0.1340
Rank 4: 0.1210
Rank 5: 0.0850
Rank 6: 0.0920
Rank 7: 0.0720
Rank 8: 0.0740
Rank 9: 0.0560
Rank 10: 0.0570


In [ ]:
distribution = (
    pd.Series(relevant_count_per_query)
    .value_counts()
    .sort_index()
)

print(distribution)

0    375
1    324
2    181
3     88
4     24
5      8
Name: count, dtype: int64


In [ ]:
recall_at_10 = (
    relevance_matrix.sum(axis=1) > 0
).mean()

print("Recall@10:", recall_at_10)

Recall@10: 0.625


In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import CrossEncoder
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

reranker = CrossEncoder(
    "BAAI/bge-reranker-base",
    device=device
)

print("Device:", device)

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Device: cuda


In [ ]:
# Retrieve the actual documents corresponding to Top-10 indices

top10_docs = corpus_df.iloc[indices.flatten()].copy()

print("Top-10 docs shape:", top10_docs.shape)

Top-10 docs shape: (10000, 5)


In [ ]:
queries_list = queries_df["text"].tolist()

reranker_pairs = []

for query_idx in range(len(queries_list)):

    query = queries_list[query_idx]

    for rank in range(10):
        doc_idx = indices[query_idx, rank]
        document = corpus_df.iloc[doc_idx]["text"]

        reranker_pairs.append([query, document])

print("Number of pairs:", len(reranker_pairs))

Number of pairs: 10000


In [ ]:
reranker_scores = reranker.predict(
    reranker_pairs,
    batch_size=16,
    show_progress_bar=True
)

print("Scores shape:", reranker_scores.shape)

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Scores shape: (10000,)


In [ ]:
reranker_scores = reranker_scores.reshape(
    len(queries_list),
    10
)

print("Reranker scores shape:", reranker_scores.shape)

Reranker scores shape: (1000, 10)


In [ ]:
reranked_order = np.argsort(
    -reranker_scores,
    axis=1
)

print("Reranked order shape:", reranked_order.shape)

Reranked order shape: (1000, 10)


In [ ]:
reranked_top10_doc_ids = np.take_along_axis(
    indices,
    reranked_order,
    axis=1
)

print("Reranked Top-10 shape:", reranked_top10_doc_ids.shape)

Reranked Top-10 shape: (1000, 10)


In [ ]:
reranked_top10_scores = np.take_along_axis(
    reranker_scores,
    reranked_order,
    axis=1
)

print("Reranked scores shape:", reranked_top10_scores.shape)

Reranked scores shape: (1000, 10)


In [ ]:
import numpy as np


def evaluate_ranking(topk_doc_ids, queries, qrels, corpus):
    """
    Evaluate ranked retrieval results using:
    Recall@10, MRR@10, nDCG@10

    topk_doc_ids:
        shape = (num_queries, 10)
        contains corpus row indices
    """

    # Convert corpus row indices → actual document IDs
    retrieved_actual_ids = corpus.iloc[topk_doc_ids.flatten()]["_id"].values
    retrieved_actual_ids = retrieved_actual_ids.reshape(topk_doc_ids.shape)

    recalls = []
    reciprocal_ranks = []
    ndcgs = []

    for i, query_id in enumerate(queries["_id"]):

        relevant_docs = set(qrels[query_id])

        retrieved_docs = retrieved_actual_ids[i]

        # -------------------------
        # Recall@10
        # -------------------------
        num_retrieved_relevant = sum(
            doc_id in relevant_docs
            for doc_id in retrieved_docs
        )

        recall = (
            num_retrieved_relevant / len(relevant_docs)
            if len(relevant_docs) > 0
            else 0.0
        )

        recalls.append(recall)

        # -------------------------
        # MRR@10
        # -------------------------
        reciprocal_rank = 0.0

        for rank, doc_id in enumerate(retrieved_docs, start=1):
            if doc_id in relevant_docs:
                reciprocal_rank = 1.0 / rank
                break

        reciprocal_ranks.append(reciprocal_rank)

        # -------------------------
        # nDCG@10
        # -------------------------
        gains = np.array([
            1 if doc_id in relevant_docs else 0
            for doc_id in retrieved_docs
        ])

        discounts = np.log2(np.arange(2, 12))

        dcg = np.sum(gains / discounts)

        ideal_gains = np.ones(
            min(len(relevant_docs), 10)
        )

        ideal_discounts = np.log2(
            np.arange(2, len(ideal_gains) + 2)
        )

        idcg = np.sum(
            ideal_gains / ideal_discounts
        )

        ndcg = dcg / idcg if idcg > 0 else 0.0

        ndcgs.append(ndcg)

    return {
        "Recall@10": np.mean(recalls),
        "MRR@10": np.mean(reciprocal_ranks),
        "nDCG@10": np.mean(ndcgs)
    }

In [ ]:
reranked_metrics = evaluate_ranking(
    reranked_top10_doc_ids,
    queries_df, # Pass queries_df instead of queries
    qrels_dict, # Pass qrels_dict instead of qrels
    corpus_df # Pass corpus_df instead of corpus
)

reranked_metrics

{'Recall@10': np.float64(0.0362564039408867),
 'MRR@10': np.float64(0.3212087301587302),
 'nDCG@10': np.float64(0.12635853117408222)}

In [ ]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Convert retrieved row indices to actual document IDs
# --------------------------------------------------

top10_doc_ids_actual = corpus_df.iloc[indices.flatten()]["_id"].values.reshape(1000, 10)

# --------------------------------------------------
# 2. Build relevance matrix using qrels
#    qrels are used ONLY for offline evaluation
# --------------------------------------------------

relevance_matrix = np.zeros((1000, 10), dtype=bool)

for i, query_id in enumerate(qrels.keys()):
    relevant_docs = set(qrels[query_id])

    for rank in range(10):
        doc_id = top10_doc_ids_actual[i, rank]
        relevance_matrix[i, rank] = doc_id in relevant_docs

# --------------------------------------------------
# 3. Calculate context statistics for different K
# --------------------------------------------------

results = []

for k in [1, 3, 5, 10]:

    relevance_topk = relevance_matrix[:, :k]

    # Average number of relevant documents retained
    avg_relevant = relevance_topk.sum(axis=1).mean()

    # Percentage of queries with at least one relevant document
    hit_rate = (relevance_topk.sum(axis=1) > 0).mean()

    # Retrieve the actual documents
    selected_doc_indices = indices[:, :k]

    selected_docs = corpus_df.iloc[selected_doc_indices.reshape(-1)]

    # Document word lengths
    doc_word_lengths = selected_docs["text"].fillna("").str.split().str.len()

    # Total words per query
    total_words_per_query = (
        doc_word_lengths
        .values
        .reshape(1000, k)
        .sum(axis=1)
    )

    results.append({
        "K": k,
        "Avg relevant docs": avg_relevant,
        "Query hit rate": hit_rate,
        "Avg context words": total_words_per_query.mean(),
        "Median context words": np.median(total_words_per_query),
        "P95 context words": np.percentile(total_words_per_query, 95)
    })

context_analysis = pd.DataFrame(results)

context_analysis

,K,Avg relevant docs,Query hit rate,Avg context words,Median context words,P95 context words
0,1,0.221,0.221,152.788,141.0,254.05
1,3,0.529,0.412,469.941,445.0,681.00
2,5,0.735,0.504,794.162,764.5,1110.35
3,10,1.086,0.625,1604.434,1549.5,2220.15


In [ ]:
import json

with open("/content/drive/MyDrive/Start_LLM/Enterprise_Semantic_Search_Engine/Data/contexts.json", "w", encoding="utf-8") as f:
    json.dump(contexts, f, ensure_ascii=False, indent=2)

print(f"Saved {len(contexts)} contexts.")

Saved 1000 contexts.


In [2]:
import json

with open("/content/drive/MyDrive/Start_LLM/Enterprise_Semantic_Search_Engine/Data/contexts.json", "r", encoding="utf-8") as f:
    contexts = json.load(f)

print(len(contexts))

1000


In [ ]:
K = 5

# Top-K documents for each query
topk_doc_indices = indices[:, :K]

def build_context(doc_indices, corpus_df):
    context_parts = []

    for rank, doc_idx in enumerate(doc_indices, start=1):
        doc_text = corpus_df.iloc[doc_idx]["text"]

        context_parts.append(
            f"[Document {rank}]\n{doc_text}"
        )

    return "\n\n".join(context_parts)


# Build context for all queries
contexts = [
    build_context(doc_indices, corpus_df)
    for doc_indices in topk_doc_indices
]

print("Number of contexts:", len(contexts))
print()
print(contexts[0])

Number of contexts: 1000

[Document 1]
Dynamic economic dispatch (DED) is one of the main functions of power generation operation and control. It determines the optimal settings of generator units with predicted load demand over a certain period of time. The objective is to operate an electric power system most economically while the system is operating within its security limits. This paper proposes a new hybrid methodology for solving DED. The proposed method is developed in such a way that a simple evolutionary programming (EP) is applied as a based level search, which can give a good direction to the optimal global region, and a local search sequential quadratic programming (SQP) is used as a fine tuning to determine the optimal solution at the final. Ten units test system with nonsmooth fuel cost function is used to illustrate the effectiveness of the proposed method compared with those obtained from EP and SQP alone.

[Document 2]
Over the last decade many metaheuristics have bee

In [ ]:
def build_prompt(query, context):
    return f"""You are a question-answering assistant.

Use the provided context to answer the question.
Base your answer only on the information contained in the context.
If the context does not contain enough information to answer the question,
say that the information is not available in the provided context.

Context:
{context}

Question:
{query}

Answer:"""

In [ ]:
prompts = [
    build_prompt(query, context)
    for query, context in zip(queries, contexts)
]

print(prompts[0])

You are a question-answering assistant.

Use the provided context to answer the question.
Base your answer only on the information contained in the context.
If the context does not contain enough information to answer the question,
say that the information is not available in the provided context.

Context:
[Document 1]
Dynamic economic dispatch (DED) is one of the main functions of power generation operation and control. It determines the optimal settings of generator units with predicted load demand over a certain period of time. The objective is to operate an electric power system most economically while the system is operating within its security limits. This paper proposes a new hybrid methodology for solving DED. The proposed method is developed in such a way that a simple evolutionary programming (EP) is applied as a based level search, which can give a good direction to the optimal global region, and a local search sequential quadratic programming (SQP) is used as a fine tuning

In [ ]:
def build_prompt(query, context):
    return f"""You are a question-answering assistant.

Use the provided context to answer the question.
Base your answer only on the information contained in the context.
If the context does not contain enough information to answer the question,
say that the information is not available in the provided context.

Context:
{context}

Question:
{query}

Answer:
"""


prompts = [
    build_prompt(query, context)
    for query, context in zip(queries_df['text'], contexts)
]

print(prompts[0])

You are a question-answering assistant.

Use the provided context to answer the question.
Base your answer only on the information contained in the context.
If the context does not contain enough information to answer the question,
say that the information is not available in the provided context.

Context:
[Document 1]
Dynamic economic dispatch (DED) is one of the main functions of power generation operation and control. It determines the optimal settings of generator units with predicted load demand over a certain period of time. The objective is to operate an electric power system most economically while the system is operating within its security limits. This paper proposes a new hybrid methodology for solving DED. The proposed method is developed in such a way that a simple evolutionary programming (EP) is applied as a based level search, which can give a good direction to the optimal global region, and a local search sequential quadratic programming (SQP) is used as a fine tuning

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [ ]:
prompt = prompts[0]

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(generated_text)

You are a question-answering assistant.

Use the provided context to answer the question.
Base your answer only on the information contained in the context.
If the context does not contain enough information to answer the question,
say that the information is not available in the provided context.

Context:
[Document 1]
Dynamic economic dispatch (DED) is one of the main functions of power generation operation and control. It determines the optimal settings of generator units with predicted load demand over a certain period of time. The objective is to operate an electric power system most economically while the system is operating within its security limits. This paper proposes a new hybrid methodology for solving DED. The proposed method is developed in such a way that a simple evolutionary programming (EP) is applied as a based level search, which can give a good direction to the optimal global region, and a local search sequential quadratic programming (SQP) is used as a fine tuning

In [ ]:
input_length = inputs["input_ids"].shape[1]

generated_tokens = outputs[0][input_length:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(answer)

The context provides a framework for understanding dynamic economic dispatch (DED) and offers a new hybrid methodology for solving it. However, it does not explicitly mention any direct search method specifically designed for solving the economic dispatch problem with valve-point effects. Therefore, the information is not directly available in the given context. Hence, the response is:

The information is not available in the provided context.


In [ ]:
def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer

In [ ]:
answer = generate_answer(prompts[0])
print(answer)

The context provides a framework for understanding dynamic economic dispatch (DED) and offers a new hybrid methodology for solving it. However, it does not explicitly mention any direct search method specifically designed for solving the economic dispatch problem with valve-point effects. Therefore, the information is not directly available in the given context. Hence, the response is:

The information is not available in the provided context.


In [ ]:
import torch

def generate_answer(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer

In [ ]:
answers = []

for i, prompt in enumerate(prompts):
    answer = generate_answer(prompt)
    answers.append(answer)

    if (i + 1) % 100 == 0:
        print(f"Generated {i + 1}/{len(prompts)} answers")

Generated 100/1000 answers
Generated 200/1000 answers
Generated 300/1000 answers
Generated 400/1000 answers
Generated 500/1000 answers
Generated 600/1000 answers
Generated 700/1000 answers


In [ ]:
print(len(answers))

NameError: name 'answers' is not defined

In [ ]:
import pickle
answers = []

for i, prompt in enumerate(prompts):

    answer = generate_answer(prompt)
    answers.append(answer)

    if (i + 1) % 50 == 0:
        with open("/content/drive/MyDrive/Start_LLM/Enterprise_Semantic_Search_Engine/Data/answers.pkl", "wb") as f:
            pickle.dump(answers, f)

        print(f"Saved {i + 1}/{len(prompts)}")

Saved 50/1000
Saved 100/1000
Saved 150/1000
Saved 200/1000
Saved 250/1000
Saved 300/1000
Saved 350/1000
Saved 400/1000
Saved 450/1000
Saved 500/1000
Saved 550/1000
Saved 600/1000
Saved 650/1000
Saved 700/1000
Saved 750/1000
Saved 800/1000
Saved 850/1000
Saved 900/1000
Saved 950/1000
Saved 1000/1000


In [3]:
import pickle

with open("/content/drive/MyDrive/Start_LLM/Enterprise_Semantic_Search_Engine/Data/answers.pkl", "rb") as f:
    answers = pickle.load(f)

print(type(answers))
print(len(answers))

<class 'list'>
1000


In [8]:
print(answers[0])

The context provides a framework for understanding dynamic economic dispatch (DED) and offers a new hybrid methodology for solving it. However, it does not explicitly mention any direct search method specifically designed for solving the economic dispatch problem with valve-point effects. Therefore, the information is not directly available in the given context. Hence, the response is:

The information is not available in the provided context.


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

JUDGE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)

judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

judge_model.eval()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [5]:
def judge_generate(prompt, max_new_tokens=256):
    inputs = judge_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(judge_model.device)

    with torch.no_grad():
        outputs = judge_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    response = judge_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response

In [6]:
CLAIM_PROMPT = """You are an evaluation assistant.

Extract all atomic factual claims from the answer below.

A claim is a factual statement that can be evaluated as true or false.

Split compound statements into separate atomic claims when appropriate.

Do not add information that is not explicitly stated in the answer.
Do not evaluate whether the claims are correct or supported.
Extract the claims exactly from the answer.

If the answer is only an abstention such as
"I don't know" or
"The information is not available in the provided context",
return an empty list.

Return valid JSON only in this format:

{{
  "claims": [
    "claim 1",
    "claim 2"
  ]
}}

Answer:
{answer}
"""

In [11]:
CLAIM_PROMPT = """Extract the factual claims from the answer below.

Write every factual claim as a separate item.

Split compound statements into separate claims.

Do not evaluate whether the claims are true or supported.
Do not add information that is not stated in the answer.

Return JSON only:

{{
  "claims": [
    "claim 1",
    "claim 2",
    "claim 3"
  ]
}}

Answer:
{answer}
"""

In [12]:
i = 0

answer = answers[i]

prompt = CLAIM_PROMPT.format(answer=answer)

raw_response = judge_generate(
    prompt,
    max_new_tokens=256
)

print(raw_response)

```json
{
  "claims": []
}
```


In [13]:
prompt = """List the factual statements in this text:

The context provides a framework for understanding dynamic economic dispatch.
The method uses a hybrid approach.

Answer only with the factual statements."""

print(judge_generate(prompt, max_new_tokens=128))

The context provides a framework for understanding dynamic economic dispatch.  
The method uses a hybrid approach.


In [14]:
SIMPLE_CLAIM_PROMPT = """Extract every factual statement from the answer below.

Write each factual statement on a separate line.

Do not judge whether the statements are correct.
Do not remove a statement because the answer later says that information is unavailable.

Answer:
{answer}

Factual statements:
"""

prompt = SIMPLE_CLAIM_PROMPT.format(answer=answers[0])

raw_response = judge_generate(
    prompt,
    max_new_tokens=256
)

print(raw_response)

1. The context provides a framework for understanding dynamic economic dispatch (DED).
2. The context offers a new hybrid methodology for solving DED.
3. The context does not explicitly mention any direct search method specifically designed for solving the economic dispatch problem with valve-point effects.
4. Therefore, the information is not directly available in the given context. To be more specific, the response should have been: "The information about a direct search method specifically designed for solving the economic dispatch problem with valve-point effects is not available in the provided context." This would make the statement more precise and accurate. 

However, since the task requires writing each factual statement on a separate line without judgment or removal based on the availability of information, I will keep the original list as requested. Here they are again:

1. The context provides a framework for understanding dynamic economic dispatch (DED).
2. The context off

In [15]:
SIMPLE_CLAIM_PROMPT = """Extract every factual claim from the answer below.

Return ONLY the factual claims.

Rules:
- Write one claim per line.
- Number each claim starting from 1.
- Do not explain your answer.
- Do not repeat any claim.
- Do not add recommendations.
- Do not rewrite or improve the claims.
- Do not add any text before or after the claims.
- Do not include abstention statements such as "The information is not available".

Answer:
{answer}
"""

In [16]:
i = 0

prompt = SIMPLE_CLAIM_PROMPT.format(answer=answers[i])

raw_response = judge_generate(
    prompt,
    max_new_tokens=256
)

print(raw_response)

Claim 1: The context provides a framework for understanding dynamic economic dispatch (DED). 

The context provides a framework for understanding dynamic economic dispatch (DED).
Claim 2: The context offers a new hybrid methodology for solving DED. 

The context offers a new hybrid methodology for solving DED.
Claim 3: The context does not explicitly mention any direct search method specifically designed for solving the economic dispatch problem with valve-point effects. 

The information is not directly available in the given context.


In [19]:
import re

def parse_claims(raw_response):
    claims = []

    # Find each "Claim N:" section
    sections = re.split(
        r"\bClaim\s+\d+\s*:\s*",
        raw_response,
        flags=re.IGNORECASE
    )

    # First element is text before Claim 1
    for section in sections[1:]:
        section = section.strip()

        # Take only the first sentence
        match = re.match(r"(.+?[.!?])(?:\s|$)", section)

        if match:
            claim = match.group(1).strip()
            claims.append(claim)

    return claims

In [20]:
claims = parse_claims(raw_response)

for i, claim in enumerate(claims, start=1):
    print(f"Claim {i}: {claim}")

Claim 1: The context provides a framework for understanding dynamic economic dispatch (DED).
Claim 2: The context offers a new hybrid methodology for solving DED.
Claim 3: The context does not explicitly mention any direct search method specifically designed for solving the economic dispatch problem with valve-point effects.


In [21]:
SUPPORT_PROMPT = """You are evaluating the faithfulness of an answer in a RAG system.

Determine whether the CLAIM is directly supported by the CONTEXT.

SUPPORTED:
The context provides sufficient evidence to support the claim.

NOT_SUPPORTED:
The context does not provide sufficient evidence to support the claim.

Do not use outside knowledge.
Do not infer information that is not supported by the context.

Return ONLY one of these two labels:

SUPPORTED
NOT_SUPPORTED

CONTEXT:
{context}

CLAIM:
{claim}

LABEL:
"""

In [22]:
context = contexts[0]
claim = claims[0]

prompt = SUPPORT_PROMPT.format(
    context=context,
    claim=claim
)

raw_support = judge_generate(
    prompt,
    max_new_tokens=16
)

print(raw_support)

SUPPORTED
NOT_SUPPORTED

NOT_SUPPORTED


In [29]:
import re


# ============================================================
# 1. Claim Extraction
# ============================================================

CLAIM_PROMPT = """Extract every factual claim from the answer below.

Return ONLY the factual claims.

Rules:
- Write one claim per line.
- Number each claim starting from 1.
- Do not explain your answer.
- Do not repeat any claim.
- Do not add recommendations.
- Do not rewrite or improve the claims.
- Do not add any text before or after the claims.
- Do not include abstention statements such as "The information is not available".

Answer:
{answer}
"""

def extract_claims(answer):
    prompt = CLAIM_PROMPT.format(answer=answer)

    raw_response = judge_generate(
        prompt,
        max_new_tokens=256
    )

    # Extract sections after "Claim N:"
    sections = re.split(
        r"\bClaim\s+\d+\s*:\s*",
        raw_response,
        flags=re.IGNORECASE
    )

    claims = []

    for section in sections[1:]:
        section = section.strip()

        # Take the first complete sentence
        match = re.match(
            r"(.+?[.!?])(?:\s|$)",
            section
        )

        if match:
            claim = match.group(1).strip()

            # Remove abstention statements if they appear
            if not re.search(
                r"\bthe information is not available\b",
                claim,
                flags=re.IGNORECASE
            ):
                claims.append(claim)

    return claims, raw_response


# ============================================================
# 2. Support Verification
# ============================================================

SUPPORT_PROMPT = """You are evaluating claims for faithfulness in a RAG system.

For each claim, determine whether the CONTEXT provides sufficient evidence to support it.

Rules:
- SUPPORTED = the context provides sufficient evidence for the claim.
- NOT_SUPPORTED = the context does not provide sufficient evidence for the claim.
- Do not use outside knowledge.
- Evaluate every claim.
- Return exactly one label for each claim.
- Do not explain your decisions.

CONTEXT:
{context}

CLAIMS:
{claims_text}

OUTPUT:
"""


def verify_support(context, claims):
    claims_text = "\n".join(
        f"Claim {i}: {claim}"
        for i, claim in enumerate(claims, start=1)
    )

    prompt = SUPPORT_PROMPT.format(
        context=context,
        claims_text=claims_text
    )

    raw_response = judge_generate(
        prompt,
        max_new_tokens=64
    )

    # Extract only the two possible verdict labels.
    verdicts = re.findall(
        r"\b(?:SUPPORTED|NOT_SUPPORTED)\b",
        raw_response.upper()
    )

    return verdicts, raw_response


# ============================================================
# 3. Faithfulness Calculation
# ============================================================

def calculate_faithfulness(claims, verdicts):

    # Safety check: every claim must have one verdict.
    if len(claims) != len(verdicts):
        raise ValueError(
            f"Number of claims ({len(claims)}) "
            f"does not match number of verdicts ({len(verdicts)})."
        )

    supported_count = sum(
        verdict == "SUPPORTED"
        for verdict in verdicts
    )

    total_claims = len(claims)

    if total_claims == 0:
        return None

    faithfulness = supported_count / total_claims

    return faithfulness


# ============================================================
# 4. Run Evaluation for One Answer
# ============================================================

answer = answers[0]
context = contexts[0]


# ---- Claim extraction ----

claims, raw_claim_response = extract_claims(answer)


# ---- Support verification ----

verdicts, raw_support_response = verify_support(
    context,
    claims
)


# ---- Faithfulness ----

faithfulness = calculate_faithfulness(
    claims,
    verdicts
)


# ============================================================
# 5. Display Results
# ============================================================

print("=" * 70)
print("ANSWER")
print("=" * 70)
print(answer)

print("\n" + "=" * 70)
print("CLAIMS")
print("=" * 70)

for i, claim in enumerate(claims, start=1):
    print(f"{i}. {claim}")

print("\n" + "=" * 70)
print("SUPPORT VERIFICATION")
print("=" * 70)

for i, (claim, verdict) in enumerate(
    zip(claims, verdicts),
    start=1
):
    print(f"Claim {i}: {verdict}")

print("\n" + "=" * 70)
print("FAITHFULNESS")
print("=" * 70)

print(f"Supported claims: {sum(v == 'SUPPORTED' for v in verdicts)}")
print(f"Total claims: {len(claims)}")

if faithfulness is not None:
    print(f"Faithfulness: {faithfulness:.4f}")
else:
    print("Faithfulness: Cannot be calculated (0 claims).")

ANSWER
The context provides a framework for understanding dynamic economic dispatch (DED) and offers a new hybrid methodology for solving it. However, it does not explicitly mention any direct search method specifically designed for solving the economic dispatch problem with valve-point effects. Therefore, the information is not directly available in the given context. Hence, the response is:

The information is not available in the provided context.

CLAIMS
1. The context provides a framework for understanding dynamic economic dispatch (DED).
2. The context offers a new hybrid methodology for solving DED.
3. The context does not explicitly mention any direct search method specifically designed for solving the economic dispatch problem with valve-point effects.

SUPPORT VERIFICATION
Claim 1: SUPPORTED
Claim 2: NOT_SUPPORTED
Claim 3: NOT_SUPPORTED

FAITHFULNESS
Supported claims: 1
Total claims: 3
Faithfulness: 0.3333


In [39]:
import re


# ============================================================
# 1. Claim Extraction
# ============================================================

CLAIM_PROMPT = """Extract every factual claim from the answer below.

Return ONLY the factual claims.

Rules:
- Write one claim per line.
- Number each claim starting from 1.
- Do not explain your answer.
- Do not repeat any claim.
- Do not add recommendations.
- Do not rewrite or improve the claims.
- Do not add any text before or after the claims.
- Do not include abstention statements such as "The information is not available".

Answer:
{answer}
"""
def extract_claims(answer):

    prompt = CLAIM_PROMPT.format(answer=answer)

    raw_response = judge_generate(
        prompt,
        max_new_tokens=256
    )

    claims = []

    # --------------------------------------------------------
    # Format 1:
    # Claim 1: ...
    # Claim 2: ...
    # --------------------------------------------------------

    sections = re.split(
        r"\bClaim\s+\d+\s*:\s*",
        raw_response,
        flags=re.IGNORECASE
    )

    if len(sections) > 1:

        for section in sections[1:]:

            section = section.strip()

            match = re.match(
                r"(.+?[.!?])(?:\s|$)",
                section
            )

            if match:

                claim = match.group(1).strip()

                if not re.search(
                    r"\bthe information is not available\b",
                    claim,
                    flags=re.IGNORECASE
                ):
                    claims.append(claim)

    # --------------------------------------------------------
    # Format 2:
    # 1. ...
    # 2. ...
    # 3. ...
    # --------------------------------------------------------

    else:

        matches = re.findall(
            r"^\s*\d+\.\s*(.+)$",
            raw_response,
            flags=re.MULTILINE
        )

        for claim in matches:

            claim = claim.strip()

            if not re.search(
                r"\bthe information is not available\b",
                claim,
                flags=re.IGNORECASE
            ):
                claims.append(claim)

    return claims, raw_response

# ============================================================
# 2. Support Verification
# ============================================================

SUPPORT_PROMPT = """You are evaluating claims for faithfulness in a RAG system.

For each claim, determine whether the CONTEXT provides sufficient evidence to support it.

Rules:
- SUPPORTED = the context provides sufficient evidence for the claim.
- NOT_SUPPORTED = the context does not provide sufficient evidence for the claim.
- Do not use outside knowledge.
- Evaluate every claim.
- Return exactly one label for each claim.
- Do not explain your decisions.

CONTEXT:
{context}

CLAIMS:
{claims_text}

OUTPUT:
"""


def verify_support(context, claims):

    claims_text = "\n".join(
        f"Claim {i}: {claim}"
        for i, claim in enumerate(claims, start=1)
    )

    prompt = SUPPORT_PROMPT.format(
        context=context,
        claims_text=claims_text
    )

    raw_response = judge_generate(
        prompt,
        max_new_tokens=64
    )

    verdicts = re.findall(
        r"\b(?:SUPPORTED|NOT_SUPPORTED)\b",
        raw_response.upper()
    )

    return verdicts, raw_response


# ============================================================
# 3. Faithfulness
# ============================================================

def calculate_faithfulness(claims, verdicts):

    if len(claims) != len(verdicts):
        return None

    if len(claims) == 0:
        return None

    supported_count = sum(
        verdict == "SUPPORTED"
        for verdict in verdicts
    )

    return supported_count / len(claims)


# ============================================================
# 4. Evaluate One Answer
# ============================================================

def evaluate_answer(answer, context):

    # ---- Claim extraction ----

    claims, raw_claim_response = extract_claims(answer)

    # ---- Support verification ----

    if len(claims) > 0:

        verdicts, raw_support_response = verify_support(
            context,
            claims
        )

    else:

        verdicts = []
        raw_support_response = ""

    # ---- Faithfulness ----

    faithfulness = calculate_faithfulness(
        claims,
        verdicts
    )

    return {
        "answer": answer,
        "context": context,
        "claims": claims,
        "verdicts": verdicts,
        "faithfulness": faithfulness,
        "raw_claim_response": raw_claim_response,
        "raw_support_response": raw_support_response
    }


# ============================================================
# 5. Run on First 10 Answers
# ============================================================

results = []

for i in range(10):

    print(f"\n{'=' * 70}")
    print(f"Evaluating Answer {i + 1}/10")
    print(f"{'=' * 70}")

    result = evaluate_answer(
        answer=answers[i],
        context=contexts[i]
    )

    results.append(result)

    print(f"Claims: {len(result['claims'])}")
    print(f"Verdicts: {len(result['verdicts'])}")
    print(f"Faithfulness: {result['faithfulness']}")


# ============================================================
# 6. Summary
# ============================================================

print("\n" + "=" * 70)
print("10-ANSWER SUMMARY")
print("=" * 70)

for i, result in enumerate(results, start=1):

    print(
        f"Answer {i}: "
        f"{len(result['claims'])} claims | "
        f"{len(result['verdicts'])} verdicts | "
        f"Faithfulness = {result['faithfulness']}"
    )


Evaluating Answer 1/10
Claims: 3
Verdicts: 3
Faithfulness: 0.3333333333333333

Evaluating Answer 2/10
Claims: 1
Verdicts: 3
Faithfulness: None

Evaluating Answer 3/10
Claims: 5
Verdicts: 5
Faithfulness: 0.2

Evaluating Answer 4/10
Claims: 5
Verdicts: 5
Faithfulness: 0.2

Evaluating Answer 5/10
Claims: 0
Verdicts: 0
Faithfulness: None

Evaluating Answer 6/10
Claims: 6
Verdicts: 6
Faithfulness: 0.6666666666666666

Evaluating Answer 7/10
Claims: 7
Verdicts: 9
Faithfulness: None

Evaluating Answer 8/10
Claims: 0
Verdicts: 0
Faithfulness: None

Evaluating Answer 9/10
Claims: 5
Verdicts: 5
Faithfulness: 0.4

Evaluating Answer 10/10
Claims: 6
Verdicts: 8
Faithfulness: None

10-ANSWER SUMMARY
Answer 1: 3 claims | 3 verdicts | Faithfulness = 0.3333333333333333
Answer 2: 1 claims | 3 verdicts | Faithfulness = None
Answer 3: 5 claims | 5 verdicts | Faithfulness = 0.2
Answer 4: 5 claims | 5 verdicts | Faithfulness = 0.2
Answer 5: 0 claims | 0 verdicts | Faithfulness = None
Answer 6: 6 claims | 6 

In [40]:
for i in [1, 4, 6, 9]:
    print("\n" + "=" * 70)
    print(f"ANSWER {i + 1} - RAW CLAIM RESPONSE")
    print("=" * 70)
    print(results[i]["raw_claim_response"])

for i in [1, 6, 9]:
    print("\n" + "=" * 70)
    print(f"ANSWER {i + 1} - RAW SUPPORT RESPONSE")
    print("=" * 70)
    print(results[i]["raw_support_response"])


ANSWER 2 - RAW CLAIM RESPONSE
1. There is no mention of bearish-bullish sentiment analysis on financial microblogs.

ANSWER 5 - RAW CLAIM RESPONSE
and document summarization. However, it does not provide specific details about the performance metrics or the exact improvements achieved in these experiments. 

In summary, the experiments described in this context highlight the potential benefits of using WordNet and automatically constructed thesauri in improving ad hoc retrieval systems, but do not offer detailed quantitative data on their effectiveness. They suggest that these tools can enhance relevance scores and reduce redundancy in search queries, thereby making information retrieval more efficient and effective. These findings are valuable for researchers interested in advancing the field of information retrieval and natural language processing. The context also notes that while these methods have been successfully applied in certain scenarios, they may require further refinement

In [41]:
import re


# ============================================================
# 1. Claim Extraction
# ============================================================

CLAIM_PROMPT = """Extract every factual claim from the answer below.

Return ONLY the factual claims.

Rules:
- Write one claim per line.
- Number each claim starting from 1.
- Do not explain your answer.
- Do not repeat any claim.
- Do not add recommendations.
- Do not rewrite or improve the claims.
- Do not add any text before or after the claims.
- Do not include abstention statements such as "The information is not available".

Answer:
{answer}
"""


def extract_claims(answer):

    prompt = CLAIM_PROMPT.format(answer=answer)

    raw_response = judge_generate(
        prompt,
        max_new_tokens=256
    )

    claims = []

    # --------------------------------------------------------
    # Format 1:
    # Claim 1: ...
    # Claim 2: ...
    # --------------------------------------------------------

    sections = re.split(
        r"\bClaim\s+\d+\s*:\s*",
        raw_response,
        flags=re.IGNORECASE
    )

    if len(sections) > 1:

        for section in sections[1:]:

            section = section.strip()

            match = re.match(
                r"(.+?[.!?])(?:\s|$)",
                section
            )

            if match:

                claim = match.group(1).strip()

                if not re.search(
                    r"\bthe information is not available\b",
                    claim,
                    flags=re.IGNORECASE
                ):
                    claims.append(claim)

    # --------------------------------------------------------
    # Format 2:
    # 1. ...
    # 2. ...
    # 3. ...
    # --------------------------------------------------------

    else:

        matches = re.findall(
            r"^\s*\d+\.\s*(.+)$",
            raw_response,
            flags=re.MULTILINE
        )

        for claim in matches:

            claim = claim.strip()

            if not re.search(
                r"\bthe information is not available\b",
                claim,
                flags=re.IGNORECASE
            ):
                claims.append(claim)

    return claims, raw_response


# ============================================================
# 2. Per-Claim Support Verification
# ============================================================

SINGLE_SUPPORT_PROMPT = """You are evaluating the faithfulness of a RAG answer.

Determine whether the CLAIM is directly supported by the CONTEXT.

SUPPORTED:
The context provides sufficient evidence to support the claim.

NOT_SUPPORTED:
The context does not provide sufficient evidence to support the claim.

Rules:
- Use only the provided context.
- Do not use outside knowledge.
- Make exactly one decision.
- Your response must begin with exactly one of:
  SUPPORTED
  NOT_SUPPORTED

CONTEXT:
{context}

CLAIM:
{claim}

DECISION:
"""


def verify_single_claim(context, claim):

    prompt = SINGLE_SUPPORT_PROMPT.format(
        context=context,
        claim=claim
    )

    raw_response = judge_generate(
        prompt,
        max_new_tokens=32
    )

    # Take the first valid verdict only.
    match = re.search(
        r"\b(NOT_SUPPORTED|SUPPORTED)\b",
        raw_response.upper()
    )

    if match:
        verdict = match.group(1)
    else:
        verdict = None

    return verdict, raw_response


# ============================================================
# 3. Verify All Claims
# ============================================================

def verify_support(context, claims):

    verdicts = []
    raw_responses = []

    for claim in claims:

        verdict, raw_response = verify_single_claim(
            context,
            claim
        )

        verdicts.append(verdict)
        raw_responses.append(raw_response)

    return verdicts, raw_responses


# ============================================================
# 4. Faithfulness
# ============================================================

def calculate_faithfulness(claims, verdicts):

    if len(claims) == 0:
        return None

    if len(claims) != len(verdicts):
        return None

    # If any claim could not be evaluated,
    # do not silently calculate a misleading score.
    if any(v is None for v in verdicts):
        return None

    supported_count = sum(
        verdict == "SUPPORTED"
        for verdict in verdicts
    )

    return supported_count / len(claims)


# ============================================================
# 5. Evaluate One Answer
# ============================================================

def evaluate_answer(answer, context):

    # ---- Claim extraction ----

    claims, raw_claim_response = extract_claims(answer)

    # ---- Support verification ----

    if len(claims) > 0:

        verdicts, raw_support_responses = verify_support(
            context,
            claims
        )

    else:

        verdicts = []
        raw_support_responses = []

    # ---- Faithfulness ----

    faithfulness = calculate_faithfulness(
        claims,
        verdicts
    )

    return {
        "answer": answer,
        "context": context,
        "claims": claims,
        "verdicts": verdicts,
        "faithfulness": faithfulness,
        "raw_claim_response": raw_claim_response,
        "raw_support_responses": raw_support_responses
    }


# ============================================================
# 6. Run on First 10 Answers
# ============================================================

results = []

for i in range(10):

    print("\n" + "=" * 70)
    print(f"Evaluating Answer {i + 1}/10")
    print("=" * 70)

    result = evaluate_answer(
        answer=answers[i],
        context=contexts[i]
    )

    results.append(result)

    print(f"Claims: {len(result['claims'])}")
    print(f"Verdicts: {len(result['verdicts'])}")
    print(f"Faithfulness: {result['faithfulness']}")


# ============================================================
# 7. Summary
# ============================================================

print("\n" + "=" * 70)
print("10-ANSWER SUMMARY")
print("=" * 70)

for i, result in enumerate(results, start=1):

    print(
        f"Answer {i}: "
        f"{len(result['claims'])} claims | "
        f"{len(result['verdicts'])} verdicts | "
        f"Faithfulness = {result['faithfulness']}"
    )


Evaluating Answer 1/10
Claims: 3
Verdicts: 3
Faithfulness: 0.3333333333333333

Evaluating Answer 2/10
Claims: 1
Verdicts: 1
Faithfulness: 0.0

Evaluating Answer 3/10
Claims: 5
Verdicts: 5
Faithfulness: 0.0

Evaluating Answer 4/10
Claims: 5
Verdicts: 5
Faithfulness: 0.0

Evaluating Answer 5/10
Claims: 0
Verdicts: 0
Faithfulness: None

Evaluating Answer 6/10
Claims: 6
Verdicts: 6
Faithfulness: 0.16666666666666666

Evaluating Answer 7/10
Claims: 7
Verdicts: 7
Faithfulness: 0.14285714285714285

Evaluating Answer 8/10
Claims: 0
Verdicts: 0
Faithfulness: None

Evaluating Answer 9/10
Claims: 5
Verdicts: 5
Faithfulness: 0.4

Evaluating Answer 10/10
Claims: 6
Verdicts: 6
Faithfulness: 0.0

10-ANSWER SUMMARY
Answer 1: 3 claims | 3 verdicts | Faithfulness = 0.3333333333333333
Answer 2: 1 claims | 1 verdicts | Faithfulness = 0.0
Answer 3: 5 claims | 5 verdicts | Faithfulness = 0.0
Answer 4: 5 claims | 5 verdicts | Faithfulness = 0.0
Answer 5: 0 claims | 0 verdicts | Faithfulness = None
Answer 6: 

In [43]:
import re


# ============================================================
# 1. Claim Extraction
# ============================================================

CLAIM_PROMPT = """Extract every factual claim from the answer below.

Return ONLY the factual claims.

Rules:
- Write one claim per line.
- Number each claim starting from 1.
- Do not explain your answer.
- Do not repeat any claim.
- Do not add recommendations.
- Do not rewrite or improve the claims.
- Do not add any text before or after the claims.
- Do not include abstention statements such as "The information is not available".

Answer:
{answer}
"""


def extract_claims(answer):

    prompt = CLAIM_PROMPT.format(answer=answer)

    raw_response = judge_generate(
        prompt,
        max_new_tokens=256
    )

    claims = []

    # --------------------------------------------------------
    # Format 1:
    # Claim 1: ...
    # Claim 2: ...
    # --------------------------------------------------------

    sections = re.split(
        r"\bClaim\s+\d+\s*:\s*",
        raw_response,
        flags=re.IGNORECASE
    )

    if len(sections) > 1:

        for section in sections[1:]:

            section = section.strip()

            match = re.match(
                r"(.+?[.!?])(?:\s|$)",
                section
            )

            if match:

                claim = match.group(1).strip()

                if not re.search(
                    r"\bthe information is not available\b",
                    claim,
                    flags=re.IGNORECASE
                ):
                    claims.append(claim)

    # --------------------------------------------------------
    # Format 2:
    # 1. ...
    # 2. ...
    # 3. ...
    # --------------------------------------------------------

    else:

        matches = re.findall(
            r"^\s*\d+\.\s*(.+)$",
            raw_response,
            flags=re.MULTILINE
        )

        for claim in matches:

            claim = claim.strip()

            if not re.search(
                r"\bthe information is not available\b",
                claim,
                flags=re.IGNORECASE
            ):
                claims.append(claim)

    return claims, raw_response


# ============================================================
# 2. Per-Claim Support Verification
# ============================================================

SINGLE_SUPPORT_PROMPT = """You are evaluating the faithfulness of a RAG answer.

Determine whether the CLAIM is directly supported by the CONTEXT.

SUPPORTED:
The context provides sufficient evidence to support the claim.

NOT_SUPPORTED:
The context does not provide sufficient evidence to support the claim.

Rules:
- Use only the provided context.
- Do not use outside knowledge.
- Make exactly one decision.
- Your response must begin with exactly one of:
  SUPPORTED
  NOT_SUPPORTED

CONTEXT:
{context}

CLAIM:
{claim}

DECISION:
"""


def verify_single_claim(context, claim):

    prompt = SINGLE_SUPPORT_PROMPT.format(
        context=context,
        claim=claim
    )

    raw_response = judge_generate(
        prompt,
        max_new_tokens=32
    )

    # Take the first valid verdict only.
    match = re.search(
        r"\b(NOT_SUPPORTED|SUPPORTED)\b",
        raw_response.upper()
    )

    if match:
        verdict = match.group(1)
    else:
        verdict = None

    return verdict, raw_response


# ============================================================
# 3. Verify All Claims
# ============================================================

def verify_support(context, claims):

    verdicts = []
    raw_responses = []

    for claim in claims:

        verdict, raw_response = verify_single_claim(
            context,
            claim
        )

        verdicts.append(verdict)
        raw_responses.append(raw_response)

    return verdicts, raw_responses


# ============================================================
# 4. Faithfulness
# ============================================================

def calculate_faithfulness(claims, verdicts):

    if len(claims) == 0:
        return None

    if len(claims) != len(verdicts):
        return None

    # If any claim could not be evaluated,
    # do not silently calculate a misleading score.
    if any(v is None for v in verdicts):
        return None

    supported_count = sum(
        verdict == "SUPPORTED"
        for verdict in verdicts
    )

    return supported_count / len(claims)


# ============================================================
# 5. Evaluate One Answer
# ============================================================

def evaluate_answer(answer, context):

    # ---- Claim extraction ----

    claims, raw_claim_response = extract_claims(answer)

    # ---- Support verification ----

    if len(claims) > 0:

        verdicts, raw_support_responses = verify_support(
            context,
            claims
        )

    else:

        verdicts = []
        raw_support_responses = []

    # ---- Faithfulness ----

    faithfulness = calculate_faithfulness(
        claims,
        verdicts
    )

    return {
        "answer": answer,
        "context": context,
        "claims": claims,
        "verdicts": verdicts,
        "faithfulness": faithfulness,
        "raw_claim_response": raw_claim_response,
        "raw_support_responses": raw_support_responses
    }


# ============================================================
# 6. Run on First 10 Answers
# ============================================================

results = []

for i in range(50):

    print("\n" + "=" * 70)
    print(f"Evaluating Answer {i + 1}/10")
    print("=" * 70)

    result = evaluate_answer(
        answer=answers[i],
        context=contexts[i]
    )

    results.append(result)

    print(f"Claims: {len(result['claims'])}")
    print(f"Verdicts: {len(result['verdicts'])}")
    print(f"Faithfulness: {result['faithfulness']}")


# ============================================================
# 7. Summary
# ============================================================

print("\n" + "=" * 70)
print("10-ANSWER SUMMARY")
print("=" * 70)

for i, result in enumerate(results, start=1):

    print(
        f"Answer {i}: "
        f"{len(result['claims'])} claims | "
        f"{len(result['verdicts'])} verdicts | "
        f"Faithfulness = {result['faithfulness']}"
    )


Evaluating Answer 1/10
Claims: 3
Verdicts: 3
Faithfulness: 0.3333333333333333

Evaluating Answer 2/10
Claims: 1
Verdicts: 1
Faithfulness: 0.0

Evaluating Answer 3/10
Claims: 5
Verdicts: 5
Faithfulness: 0.0

Evaluating Answer 4/10
Claims: 5
Verdicts: 5
Faithfulness: 0.0

Evaluating Answer 5/10
Claims: 0
Verdicts: 0
Faithfulness: None

Evaluating Answer 6/10
Claims: 6
Verdicts: 6
Faithfulness: 0.16666666666666666

Evaluating Answer 7/10
Claims: 7
Verdicts: 7
Faithfulness: 0.14285714285714285

Evaluating Answer 8/10
Claims: 0
Verdicts: 0
Faithfulness: None

Evaluating Answer 9/10
Claims: 5
Verdicts: 5
Faithfulness: 0.4

Evaluating Answer 10/10
Claims: 6
Verdicts: 6
Faithfulness: 0.0

Evaluating Answer 11/10
Claims: 3
Verdicts: 3
Faithfulness: 0.3333333333333333

Evaluating Answer 12/10
Claims: 9
Verdicts: 9
Faithfulness: 0.3333333333333333

Evaluating Answer 13/10
Claims: 3
Verdicts: 3
Faithfulness: 0.0

Evaluating Answer 14/10
Claims: 6
Verdicts: 6
Faithfulness: 0.16666666666666666

Eva

In [44]:
# Only answers with a valid Faithfulness score
valid_scores = [
    r["faithfulness"]
    for r in results
    if r["faithfulness"] is not None
]

# Mean answer-level Faithfulness
mean_faithfulness = sum(valid_scores) / len(valid_scores)

print("=" * 70)
print("50-ANSWER FAITHFULNESS SUMMARY")
print("=" * 70)

print(f"Total answers: {len(results)}")
print(f"Evaluated answers: {len(valid_scores)}")
print(f"Abstentions / no claims: {len(results) - len(valid_scores)}")
print(f"Mean Faithfulness: {mean_faithfulness:.4f}")

50-ANSWER FAITHFULNESS SUMMARY
Total answers: 50
Evaluated answers: 42
Abstentions / no claims: 8
Mean Faithfulness: 0.0935


In [44]:
# ============================================================
# Context Recall — 10 Queries
# ============================================================

K = 5
N = 10

context_recall_results = []

for i in range(N):

    print("=" * 70)
    print(f"Evaluating Query {i + 1}/{N}")
    print("=" * 70)

    # --------------------------------------------------------
    # 1. Query ID
    # --------------------------------------------------------
    query_id = queries_df.iloc[i]["_id"]

    # --------------------------------------------------------
    # 2. Ground-truth relevant documents from qrels
    # --------------------------------------------------------
    relevant_doc_ids = set(
        qrels_df[
            qrels_df["query_id"] == query_id
        ]["doc_id"]
    )

    # --------------------------------------------------------
    # 3. Retrieved Top-K documents
    # --------------------------------------------------------
    retrieved_indices = top10_doc_ids[i, :K]

    retrieved_doc_ids = set(
        corpus.iloc[retrieved_indices]["_id"]
    )

    # --------------------------------------------------------
    # 4. Calculate overlap
    # --------------------------------------------------------
    retrieved_relevant = (
        relevant_doc_ids & retrieved_doc_ids
    )

    num_relevant = len(relevant_doc_ids)
    num_retrieved_relevant = len(retrieved_relevant)

    # --------------------------------------------------------
    # 5. Context Recall
    # --------------------------------------------------------
    if num_relevant > 0:
        context_recall = (
            num_retrieved_relevant / num_relevant
        )
    else:
        context_recall = None

    print(f"Relevant documents: {num_relevant}")
    print(f"Relevant retrieved in context: {num_retrieved_relevant}")
    print(f"Context Recall: {context_recall}")

    context_recall_results.append({
        "query_index": i,
        "query_id": query_id,
        "num_relevant": num_relevant,
        "num_retrieved_relevant": num_retrieved_relevant,
        "context_recall": context_recall
    })


# ============================================================
# Summary
# ============================================================

valid_scores = [
    r["context_recall"]
    for r in context_recall_results
    if r["context_recall"] is not None
]

mean_context_recall = (
    sum(valid_scores) / len(valid_scores)
    if valid_scores
    else None
)

print("\n" + "=" * 70)
print("10-QUERY CONTEXT RECALL SUMMARY")
print("=" * 70)

print(f"Queries: {len(context_recall_results)}")
print(f"Mean Context Recall: {mean_context_recall:.4f}")

relevant answers

In [50]:
import re

# ============================================================
# Answer Relevance Evaluation
# ============================================================

def evaluate_answer_relevance(question, answer):

    prompt = f"""
You are evaluating the relevance of an answer to a question.

Evaluate ONLY whether the answer directly addresses the question.

Scoring:
0 = Completely irrelevant
1 = Slightly relevant, but mostly does not answer the question
2 = Mostly relevant and addresses the main question
3 = Highly relevant and directly answers the question

Rules:
- Focus only on relevance.
- Do NOT judge factual correctness.
- Do NOT judge faithfulness to any context.
- Do NOT use outside information.
- Return exactly one integer: 0, 1, 2, or 3.

QUESTION:
{question}

ANSWER:
{answer}

SCORE:
"""

    raw_response = judge_generate(
        prompt,
        max_new_tokens=10
    )

    # Extract the first standalone 0-3 score
    match = re.search(r"\b([0-3])\b", raw_response)

    if match:
        score = int(match.group(1))
    else:
        score = None

    return score, raw_response


# ============================================================
# Evaluate first 10 answers
# ============================================================

N = 10

relevance_results = []

for i in range(N):

    print("=" * 70)
    print(f"Evaluating Answer Relevance {i + 1}/{N}")
    print("=" * 70)

    question = queries_df.iloc[i]["text"]
    answer = answers[i]

    score, raw_response = evaluate_answer_relevance(
        question,
        answer
    )

    print(f"Relevance Score: {score}")

    relevance_results.append({
        "answer_index": i,
        "question": question,
        "answer": answer,
        "score": score,
        "raw_response": raw_response
    })


# ============================================================
# Summary
# ============================================================

valid_scores = [
    r["score"]
    for r in relevance_results
    if r["score"] is not None
]

mean_relevance = (
    sum(valid_scores) / len(valid_scores)
    if valid_scores
    else None
)

print("\n" + "=" * 70)
print("10-ANSWER RELEVANCE SUMMARY")
print("=" * 70)

print(f"Total answers: {len(relevance_results)}")
print(f"Evaluated answers: {len(valid_scores)}")
print(f"Mean Answer Relevance: {mean_relevance:.4f}")

Evaluating Answer Relevance 1/10
Relevance Score: 0
Evaluating Answer Relevance 2/10
Relevance Score: 3
Evaluating Answer Relevance 3/10
Relevance Score: 3
Evaluating Answer Relevance 4/10
Relevance Score: 3
Evaluating Answer Relevance 5/10
Relevance Score: 3
Evaluating Answer Relevance 6/10
Relevance Score: 3
Evaluating Answer Relevance 7/10
Relevance Score: 3
Evaluating Answer Relevance 8/10
Relevance Score: 3
Evaluating Answer Relevance 9/10
Relevance Score: 3
Evaluating Answer Relevance 10/10
Relevance Score: 3

10-ANSWER RELEVANCE SUMMARY
Total answers: 10
Evaluated answers: 10
Mean Answer Relevance: 2.7000


correctness

In [51]:
import re

def evaluate_correctness(question, reference, answer):
    prompt = f"""
You are evaluating the correctness of a generated answer.

Evaluate how factually correct the GENERATED ANSWER is compared
with the REFERENCE ANSWER.

Scoring:
0 = Completely incorrect
1 = Mostly incorrect, with major errors
2 = Mostly correct, but has minor errors or omissions
3 = Fully correct

Rules:
- Compare the generated answer against the reference answer.
- Focus only on factual correctness.
- Do NOT judge relevance.
- Do NOT judge faithfulness to any retrieved context.
- Do NOT use outside information.
- Return exactly one integer: 0, 1, 2, or 3.

QUESTION:
{question}

REFERENCE ANSWER:
{reference}

GENERATED ANSWER:
{answer}

SCORE:
"""

    raw_response = judge_generate(prompt, max_new_tokens=10)

    match = re.search(r"\b([0-3])\b", raw_response)

    if match:
        score = int(match.group(1))
    else:
        score = None

    return score, raw_response

In [ ]:
N = 10

correctness_results = []

for i in range(N):

    print("=" * 70)
    print(f"Evaluating Correctness {i + 1}/{N}")
    print("=" * 70)

    question = queries_df.iloc[i]["text"]
    reference = references[i]
    answer = answers[i]

    score, raw_response = evaluate_correctness(
        question,
        reference,
        answer
    )

    print(f"Correctness Score: {score}")

    correctness_results.append({
        "answer_index": i,
        "question": question,
        "reference": reference,
        "answer": answer,
        "score": score,
        "raw_response": raw_response
    })

In [ ]:
ABSTENTION_PATTERNS = [
    "information is not available in the provided context",
    "not available in the provided context",
    "cannot be determined from the provided context",
    "not enough information in the provided context",
    "the context does not provide enough information",
    "i don't know",
    "i do not know"
]


def is_abstention(answer):

    answer_lower = answer.lower().strip()

    return any(
        pattern in answer_lower
        for pattern in ABSTENTION_PATTERNS
    )

In [ ]:
i = 0

query = queries_df.iloc[i]['text']
context = contexts[i]
answer = answers[i]

print("QUERY:\n", query)
print("\nANSWER:\n", answer)
print("\nCONTEXT:\n", context)

QUERY:
 A Direct Search Method to solve Economic Dispatch Problem with Valve-Point Effect

ANSWER:
 The context provides a framework for understanding dynamic economic dispatch (DED) and offers a new hybrid methodology for solving it. However, it does not explicitly mention any direct search method specifically designed for solving the economic dispatch problem with valve-point effects. Therefore, the information is not directly available in the given context. Hence, the response is:

The information is not available in the provided context.

CONTEXT:
 [Document 1]
Dynamic economic dispatch (DED) is one of the main functions of power generation operation and control. It determines the optimal settings of generator units with predicted load demand over a certain period of time. The objective is to operate an electric power system most economically while the system is operating within its security limits. This paper proposes a new hybrid methodology for solving DED. The proposed method i